[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prefect-certified/notebooks/day-12-docker-kubernetes.ipynb#scrollTo=a3b4c5d6)

---
# Day 12 · Docker and Kubernetes Infrastructure for Production Flows
**certified-journeys / prefect-certified** · Practice

> **Goal for today:** Package a Prefect flow into a Docker image, create a Docker work pool, deploy a flow against it, and understand how Kubernetes work pools create per-run Job resources for production-scale orchestration.


In [ ]:
%pip install -q "prefect>=2.14" prefect-docker


---
## Step 1 · Why containerise Prefect flows?

Running flows in Docker containers solves the three most common production reliability problems:

| Problem (without containers) | Solution (with containers) |
|---|---|
| "Works on my laptop, fails on the worker" | Image pins exact Python + dependency versions |
| Dependency conflicts between flows | Each flow has its own isolated environment |
| Worker needs Python reinstall after upgrade | Worker only needs Docker — no Python env management |
| Hard to reproduce a failing run | Pin the image tag → same environment forever |

**Container vs process work pool:**

| Feature | `process` pool | `docker` pool |
|---|---|---|
| Isolation | None (shares worker env) | Full — each run in fresh container |
| Dependency management | Worker Python env | Image build |
| Startup overhead | Near-zero | 1–5 s (pull + start) |
| Best for | Local dev, CI | Staging and production |

**Image tag discipline — the single most important rule:**
- **Development:** `:latest` is acceptable (builds always overwrite it)
- **Production:** NEVER use `:latest` — pin to a content hash or semantic version
  - `my-flow:1.4.2` → deterministic, rollback in one command
  - `my-flow:sha-a3f9b1c` → tied to the exact git commit


In [ ]:
# Write a complete, production-quality Dockerfile for a Prefect flow
# and inspect every instruction's purpose

import pathlib
import tempfile
import textwrap

# Create a temporary directory to hold our project files
project_dir = pathlib.Path(tempfile.mkdtemp())
print(f"Project directory: {project_dir}")

# ── 1. Write requirements.txt ─────────────────────────────────────────────────
requirements_txt = textwrap.dedent("""
    # Pin every dependency to an exact version — reproducibility over convenience
    prefect==2.14.11
    pandas==2.1.4
    pyarrow==14.0.2
    requests==2.31.0
""").strip()

(project_dir / "requirements.txt").write_text(requirements_txt)

# ── 2. Write the Dockerfile ───────────────────────────────────────────────────
dockerfile_content = textwrap.dedent("""
    # syntax=docker/dockerfile:1

    # ── Stage 1: dependency installation ─────────────────────────────────────
    # Use an explicit digest or full semver tag — never 'latest'
    FROM python:3.11.7-slim AS builder

    WORKDIR /build

    # Install only what pip needs — system packages first, then Python packages
    # --no-cache-dir reduces image size; --upgrade pip avoids known security issues
    RUN pip install --upgrade pip --no-cache-dir

    COPY requirements.txt .

    # Install into /install so we can copy it to the final stage cleanly
    RUN pip install --no-cache-dir --prefix=/install -r requirements.txt

    # ── Stage 2: runtime image (final, smaller) ───────────────────────────────
    FROM python:3.11.7-slim AS runtime

    # Non-root user — never run flow processes as root
    RUN useradd --create-home --uid 1001 prefect
    WORKDIR /app

    # Copy installed packages from builder stage (no pip, no build tools in final image)
    COPY --from=builder /install /usr/local

    # Copy flow code last — it changes most often (maximizes cache hit on deps layer)
    COPY --chown=prefect:prefect flow.py .

    USER prefect

    # The Prefect worker starts the container and calls the entrypoint
    # No CMD needed — the docker work pool injects the run command at startup
""").strip()

(project_dir / "Dockerfile").write_text(dockerfile_content)

# ── 3. Write the flow module ──────────────────────────────────────────────────
flow_code = textwrap.dedent("""
    from prefect import flow, task, get_run_logger
    import pandas as pd

    @task
    def ingest(source: str, rows: int) -> pd.DataFrame:
        logger = get_run_logger()
        logger.info(f"Ingesting {rows} rows from '{source}'")
        data = {"id": range(rows), "source": source, "value": [i * 3.14 for i in range(rows)]}
        return pd.DataFrame(data)

    @task
    def transform(df: pd.DataFrame, multiplier: float) -> pd.DataFrame:
        df = df.copy()
        df["value"] = (df["value"] * multiplier).round(4)
        return df

    @flow(name="docker-flow", log_prints=True)
    def docker_flow(
        source: str = "prod-api",
        rows: int = 100,
        multiplier: float = 1.0,
    ) -> dict:
        df = ingest(source=source, rows=rows)
        df = transform(df=df, multiplier=multiplier)
        summary = {"source": source, "rows": len(df), "total": df['value'].sum().round(4)}
        print(f"Done: {summary}")
        return summary

    if __name__ == "__main__":
        docker_flow()
""").strip()

(project_dir / "flow.py").write_text(flow_code)

print("\nProject files created:")
for f in sorted(project_dir.iterdir()):
    size = f.stat().st_size
    print(f"  {f.name:<22} {size:>6} bytes")

print("\nDockerfile content:")
print(dockerfile_content)


**What just happened?**
- **Multi-stage build** separates dependency installation from the runtime image — the final image has no compiler or build tools, reducing attack surface and size
- **`COPY flow.py .` goes last** — the dependency layer is cached by Docker unless `requirements.txt` changes; moving code changes don't invalidate the expensive pip install layer
- **Non-root user** (`prefect` with UID 1001) is a security best practice — Kubernetes pod security policies often reject root-running containers
- No `CMD` is needed — the Prefect Docker work pool injects the start command (`python -m prefect worker start`) at container start


In [ ]:
# Simulate `docker build` and `docker tag` — show the commands and validate inputs
# (Actual Docker build runs outside Colab — we generate and validate the commands)

import shlex
import re

# ── Image naming convention helper ────────────────────────────────────────────
def build_image_tags(
    registry: str,
    repo: str,
    semver: str,
    git_sha: str,
) -> dict[str, str]:
    """Generate the three canonical image tag variants for a flow image."""
    # Validate semver format
    if not re.match(r'^\d+\.\d+\.\d+$', semver):
        raise ValueError(f"semver must be X.Y.Z format, got: {semver!r}")
    # Validate git SHA (7–40 hex chars)
    if not re.match(r'^[0-9a-f]{7,40}$', git_sha):
        raise ValueError(f"git_sha must be 7–40 hex chars, got: {git_sha!r}")

    base = f"{registry}/{repo}"
    return {
        "semver":    f"{base}:{semver}",                    # e.g. 1.4.2
        "sha":       f"{base}:sha-{git_sha[:8]}",           # e.g. sha-a3f9b1c0
        "major":     f"{base}:{semver.split('.')[0]}",      # e.g. 1  (mutable alias)
    }

tags = build_image_tags(
    registry="us-central1-docker.pkg.dev/my-project",
    repo="flows/my-flow",
    semver="1.4.2",
    git_sha="a3f9b1c0d2e4f5a6",
)

print("Image tags to create:")
for label, tag in tags.items():
    print(f"  {label:<10} {tag}")

# Generate the shell commands
print("\nBuild & tag commands:")
print(f"  # Build with semver tag")
print(f"  docker build -t {tags['semver']} .")
print(f"")
print(f"  # Add additional tags (no rebuild — just alias the image ID)")
print(f"  docker tag {tags['semver']} {tags['sha']}")
print(f"  docker tag {tags['semver']} {tags['major']}")
print(f"")
print(f"  # Push all three tags")
for tag in tags.values():
    print(f"  docker push {tag}")

print("\nLayer cache strategy:")
layers = [
    ("FROM python:3.11.7-slim",              "Base OS — cached forever unless Python version changes"),
    ("RUN pip install ... requirements.txt", "Dependencies — cached until requirements.txt changes"),
    ("COPY flow.py .",                       "Flow code — invalidated on every code change"),
]
for i, (layer, desc) in enumerate(layers, 1):
    print(f"  Layer {i}: {layer}")
    print(f"           → {desc}")


**What just happened?**
- `build_image_tags()` enforces the tagging convention — semver, SHA alias, and major alias — and validates inputs
- `docker tag` creates additional aliases without rebuilding — a single `docker build` plus multiple `docker tag` calls is the correct pattern
- **Layer ordering matters**: base → dependencies → code ensures the expensive dependency installation is cached across code-only changes
- The SHA tag (`sha-a3f9b1c0`) is the most important for debugging — it ties the running container to an exact git commit


In [ ]:
# Create and inspect a Docker work pool configuration
# In a real environment: prefect work-pool create my-docker-pool --type docker

import json

# The base_job_template is what the Docker work pool uses to configure each container run
# This JSON structure mirrors what Prefect stores in the work pool record
docker_work_pool_config = {
    "name": "my-docker-pool",
    "type": "docker",
    "base_job_template": {
        "job_configuration": {
            # The image to run — override per-deployment
            "image": "{{ defaults.image }}",
            # Environment variables injected into every container
            "env": {
                "PREFECT_API_URL": "{{ defaults.api_url }}",
                "PREFECT_API_KEY": "{{ defaults.api_key }}",
            },
            # Network mode: 'host' for local dev, 'bridge' for isolated prod
            "network_mode": "bridge",
            # Auto-remove container after run completes
            "auto_remove": True,
            # Resource limits
            "mem_limit": "2g",
            "cpu_quota": 100000,   # 100ms out of 100ms period = 1 CPU
            "labels": {
                "managed-by": "prefect",
                "work-pool": "my-docker-pool",
            },
        },
        "variables": {
            "type": "object",
            "properties": {
                "image":   {"type": "string", "title": "Docker image",  "default": "prefecthq/prefect:2-python3.11"},
                "api_url": {"type": "string", "title": "Prefect API URL"},
                "api_key": {"type": "string", "title": "Prefect API Key", "format": "password"},
            },
        },
    },
    "description": "Docker work pool for production flow runs",
    "concurrency_limit": 10,   # max 10 containers running simultaneously
}

print("Docker work pool configuration:")
print(json.dumps(docker_work_pool_config, indent=2))

print()
print("CLI creation command:")
print("  prefect work-pool create my-docker-pool --type docker")
print()
print("Start a Docker worker (polls the pool and runs containers):")
print("  prefect worker start --pool my-docker-pool --type docker")
print()
print("Key configuration fields:")
fields = [
    ("type",              "'docker' — tells Prefect which worker implementation to use"),
    ("image",             "Docker image to run — override per deployment"),
    ("auto_remove",       "True = container deleted after run (no orphan containers)"),
    ("mem_limit",         "Hard memory limit — container is OOM-killed if exceeded"),
    ("concurrency_limit", "Max concurrent containers — prevents resource exhaustion"),
    ("network_mode",      "'host' for local dev (access localhost), 'bridge' for prod isolation"),
]
for field, desc in fields:
    print(f"  {field:<22} {desc}")


**What just happened?**
- The `base_job_template` defines the **defaults** for every run in this pool — deployments can override individual fields
- `auto_remove: True` is critical in production — without it, stopped containers accumulate and consume disk space
- `mem_limit` and `cpu_quota` prevent a runaway flow from exhausting the host's resources
- `concurrency_limit` at the pool level throttles total container count — set this based on your host's RAM


In [ ]:
# Deploy a flow to the Docker work pool — show the full deployment configuration
# and simulate what the Docker worker does at runtime

from prefect import flow, task, get_run_logger
from prefect.testing.utilities import prefect_test_harness
import json

# ── Flow definition (matches what would be in flow.py inside the Docker image) ─

@task(retries=2, retry_delay_seconds=5)
def fetch_records(source: str, limit: int) -> list[dict]:
    logger = get_run_logger()
    logger.info(f"Fetching {limit} records from {source!r}")
    return [{"id": i, "source": source, "value": round(i * 2.718, 3)} for i in range(limit)]

@task
def aggregate(records: list[dict]) -> dict:
    total = sum(r["value"] for r in records)
    return {"count": len(records), "total": round(total, 4)}

@flow(name="docker-deployed-flow", log_prints=True)
def docker_deployed_flow(
    source: str = "prod-api",
    limit: int = 50,
) -> dict:
    """Flow that runs in a Docker container via the Docker work pool."""
    records = fetch_records(source=source, limit=limit)
    summary = aggregate(records=records)
    print(f"Aggregation complete: {summary}")
    return summary

# ── Show the deployment configuration ────────────────────────────────────────
# This mirrors what `flow.from_source().deploy()` would generate
deployment_config = {
    "name": "docker-deployed-flow/prod",
    "flow_name": "docker-deployed-flow",
    "entrypoint": "flow.py:docker_deployed_flow",   # path inside the Docker image
    "work_pool_name": "my-docker-pool",
    "parameters": {"source": "prod-api", "limit": 50},
    "tags": ["env:prod", "team:data-eng"],
    "version": "1.4.2",
    "job_variables": {
        # Override the pool default image for this specific deployment
        "image": "us-central1-docker.pkg.dev/my-project/flows/my-flow:1.4.2",
        "mem_limit": "4g",     # this flow needs more memory than the pool default
    },
    "schedule": {"cron": "0 6 * * *", "timezone": "UTC"},   # daily at 06:00 UTC
    "description": "Production flow running in a pinned Docker image",
}

print("Deployment configuration (what goes to Prefect server):")
print(json.dumps(deployment_config, indent=2))

# ── Run the flow locally (same code, no Docker needed in Colab) ───────────────
print()
print("Local test run (production runs in Docker container):")
with prefect_test_harness():
    result = docker_deployed_flow(source="staging", limit=15)
    print(f"Result: {result}")

# ── Show what the Docker worker does at runtime ───────────────────────────────
print()
print("What the Docker worker executes at run time:")
container_cmd = [
    "docker", "run",
    "--rm",                                                          # auto_remove
    "--env", "PREFECT_API_URL=https://api.prefect.cloud/api/...",
    "--env", "PREFECT_API_KEY=pnu_...",
    "--memory", "4g",
    "--cpus", "1",
    "--network", "bridge",
    "--label", "managed-by=prefect",
    "us-central1-docker.pkg.dev/my-project/flows/my-flow:1.4.2",    # pinned image
    "python", "-m", "prefect.engine",                               # Prefect entrypoint
]
print("  " + " \\\n    ".join(container_cmd))


**What just happened?**
- `job_variables` at the deployment level **override** the work pool defaults — the pool is the floor, the deployment is the override
- `entrypoint: "flow.py:docker_deployed_flow"` tells the worker where inside the image to find the flow function
- The Docker worker constructs the `docker run` command, injecting `PREFECT_API_URL` and `PREFECT_API_KEY` — the container authenticates and reports its state back to the server
- `--rm` flag (auto_remove) ensures the container is deleted when done — without this, stopped containers accumulate and fill disk


---
## Step 4 · Kubernetes work pools — architecture overview

A Kubernetes (K8s) work pool takes the Docker pattern further: instead of running a container on a single worker machine, Prefect creates a **Kubernetes Job** per flow run. The Job runs in your cluster — auto-scaled, load-balanced, and observable.

**Component map:**

```
Prefect Cloud / Server
        ↓  (HTTP polling, every N seconds)
Prefect Worker (long-running K8s Deployment)
        ↓  (creates a K8s Job via the Kubernetes API)
K8s Job → Pod → Container (runs the flow)
        ↓  (flow run metadata sent back to Prefect API)
Prefect Cloud / Server
```

**Key difference from Docker pool:**

| Aspect | Docker pool | Kubernetes pool |
|---|---|---|
| Infrastructure | Single VM/machine | Kubernetes cluster |
| Scheduling | Docker daemon | K8s scheduler (bin packing, node affinity) |
| Scaling | Limited to host resources | Auto-scale via HPA / cluster autoscaler |
| Per-run isolation | Docker container | Kubernetes Pod (stronger isolation) |
| Resource quotas | `--memory`, `--cpus` | K8s `resources.requests/limits` |
| Observability | `docker logs` | `kubectl logs`, Prometheus, Grafana |


In [ ]:
# Model the Kubernetes Job manifest that the K8s work pool creates per flow run
# In production this is generated by the Prefect worker and submitted to the K8s API

import json
import uuid

def generate_k8s_job_manifest(
    flow_run_name: str,
    image: str,
    api_url: str,
    api_key_secret_name: str,
    namespace: str = "prefect-flows",
    cpu_request: str = "500m",
    memory_request: str = "512Mi",
    cpu_limit: str = "2000m",
    memory_limit: str = "4Gi",
) -> dict:
    """Generate the K8s Job manifest for a Prefect flow run.

    The Prefect Kubernetes worker creates this manifest programmatically
    and submits it to the cluster via the Kubernetes API.
    """
    # Kubernetes label values must be alphanumeric + dash, max 63 chars
    safe_name = flow_run_name.replace("_", "-").lower()[:63]

    return {
        "apiVersion": "batch/v1",
        "kind": "Job",
        "metadata": {
            "name": f"prefect-{safe_name}",
            "namespace": namespace,
            "labels": {
                "managed-by": "prefect",
                "prefect.io/flow-run-name": safe_name,
            },
            "annotations": {
                "prefect.io/flow-run-id": str(uuid.uuid4()),  # links Job back to Prefect run
            },
        },
        "spec": {
            "backoffLimit": 0,          # Prefect handles retries — don't let K8s retry too
            "ttlSecondsAfterFinished": 3600,  # clean up completed Jobs after 1 hour
            "template": {
                "metadata": {
                    "labels": {"managed-by": "prefect"},
                },
                "spec": {
                    "restartPolicy": "Never",  # matches backoffLimit=0
                    "serviceAccountName": "prefect-worker-sa",
                    "containers": [
                        {
                            "name": "prefect-flow",
                            "image": image,
                            "imagePullPolicy": "IfNotPresent",  # pull only if tag absent locally
                            "command": ["python", "-m", "prefect.engine"],
                            "env": [
                                {"name": "PREFECT_API_URL", "value": api_url},
                                {
                                    "name": "PREFECT_API_KEY",
                                    "valueFrom": {
                                        "secretKeyRef": {
                                            "name": api_key_secret_name,  # K8s Secret name
                                            "key": "PREFECT_API_KEY",      # key within the Secret
                                        }
                                    },
                                },
                            ],
                            "resources": {
                                "requests": {"cpu": cpu_request, "memory": memory_request},
                                "limits":   {"cpu": cpu_limit,   "memory": memory_limit},
                            },
                            "securityContext": {
                                "runAsNonRoot": True,
                                "runAsUser": 1001,
                                "readOnlyRootFilesystem": True,
                                "allowPrivilegeEscalation": False,
                            },
                        }
                    ],
                },
            },
        },
    }

manifest = generate_k8s_job_manifest(
    flow_run_name="daily-ingest-prod-2024-01-15",
    image="us-central1-docker.pkg.dev/my-project/flows/my-flow:1.4.2",
    api_url="https://api.prefect.cloud/api/accounts/acct-xxx/workspaces/ws-yyy",
    api_key_secret_name="prefect-api-key",
    memory_limit="8Gi",
)

print("Kubernetes Job manifest (generated per flow run):")
print(json.dumps(manifest, indent=2))


**What just happened?**
- `backoffLimit: 0` is intentional — Kubernetes would create a *new* Pod on retry, losing all in-process state. Let Prefect handle retries via `@task(retries=N)` instead
- `ttlSecondsAfterFinished: 3600` auto-cleans up completed Jobs — without this, etcd fills up with dead Job records
- The API key is read from a **Kubernetes Secret** (`secretKeyRef`), not stored in the manifest itself — the manifest can be committed to git safely
- `securityContext` enforces non-root execution and a read-only filesystem — required by most enterprise K8s security policies


In [ ]:
# Simulate the Kubernetes work pool configuration
# and show the CLI commands to create and manage it

import json

k8s_work_pool_config = {
    "name": "my-k8s-pool",
    "type": "kubernetes",
    "base_job_template": {
        "job_configuration": {
            "namespace": "prefect-flows",
            "image": "{{ defaults.image }}",
            "service_account_name": "prefect-worker-sa",
            "finished_job_ttl": 3600,
            "job_watch_timeout_seconds": 300,
            "pod_watch_timeout_seconds": 120,
            "stream_output": True,
            # customise_job_manifest allows Jinja2 overrides per deployment
            "customizations": [],
        },
        "variables": {
            "type": "object",
            "properties": {
                "image": {
                    "type": "string",
                    "title": "Container image",
                    "default": "prefecthq/prefect:2-python3.11",
                }
            },
        },
    },
    "concurrency_limit": 50,  # max 50 K8s Jobs simultaneously
    "description": "Kubernetes work pool for production flows",
}

print("Kubernetes work pool configuration:")
print(json.dumps(k8s_work_pool_config, indent=2))

print()
print("CLI setup sequence:")
setup_steps = [
    ("1", "Create a Kubernetes namespace for flow runs",
     "kubectl create namespace prefect-flows"),
    ("2", "Create a K8s Secret for the Prefect API key",
     "kubectl create secret generic prefect-api-key \\\n"
     "    --from-literal=PREFECT_API_KEY=pnu_... \\\n"
     "    -n prefect-flows"),
    ("3", "Create the Kubernetes work pool in Prefect",
     "prefect work-pool create my-k8s-pool --type kubernetes"),
    ("4", "Start the Prefect Kubernetes worker (runs as a K8s Deployment)",
     "prefect worker start --pool my-k8s-pool --type kubernetes"),
    ("5", "Deploy a flow targeting the K8s pool",
     "prefect deploy flow.py:docker_deployed_flow \\\n"
     "    --name k8s-prod \\\n"
     "    --pool my-k8s-pool \\\n"
     "    --image us-central1-docker.pkg.dev/my-project/flows/my-flow:1.4.2"),
    ("6", "Monitor Jobs as they run",
     "kubectl get jobs -n prefect-flows -w"),
    ("7", "View logs for a specific Job",
     "kubectl logs -n prefect-flows job/prefect-daily-ingest-prod"),
]
for step_num, desc, cmd in setup_steps:
    print(f"  Step {step_num}: {desc}")
    for line in cmd.split("\n"):
        print(f"    {line}")
    print()


**What just happened?**
- The K8s work pool worker is itself a long-running Kubernetes Deployment — it polls Prefect and submits Jobs
- `job_watch_timeout_seconds` and `pod_watch_timeout_seconds` determine how long the worker waits for Kubernetes to schedule the Pod before declaring a timeout
- `stream_output: True` pipes Pod stdout back to the Prefect worker, which forwards logs to the Prefect API — visible in the UI
- `kubectl get jobs -n prefect-flows -w` gives live Job state from the Kubernetes control plane — a useful operational complement to the Prefect UI


In [ ]:
# Full end-to-end flow test using the process pool equivalent (Colab-safe)
# Demonstrates the identical flow that runs in Docker/K8s in production

from prefect import flow, task, get_run_logger
from prefect.testing.utilities import prefect_test_harness
import json

@task(retries=2, retry_delay_seconds=1)
def extract(source: str, batch_size: int) -> list[dict]:
    """Extract records from the source API.

    Production: replaces with requests.get(api_url, params={...}).json()
    """
    logger = get_run_logger()
    logger.info(f"Extracting batch of {batch_size} from '{source}'")
    return [{"id": i, "source": source, "raw": i * 1.414} for i in range(batch_size)]

@task
def transform_records(records: list[dict], scale: float) -> list[dict]:
    """Apply scaling and type coercion."""
    return [
        {**r, "processed": round(r["raw"] * scale, 4), "valid": r["raw"] > 0}
        for r in records
    ]

@task
def load_to_sink(records: list[dict], destination: str) -> int:
    """Load records to the destination.

    Production: replaces with BigQuery / PostgreSQL / S3 write.
    """
    logger = get_run_logger()
    valid = [r for r in records if r["valid"]]
    logger.info(f"Loaded {len(valid)}/{len(records)} valid records to '{destination}'")
    return len(valid)

@flow(name="container-etl-pipeline", log_prints=True)
def container_etl_pipeline(
    source: str = "prod-api",
    destination: str = "prod-warehouse.events",
    batch_size: int = 100,
    scale: float = 1.0,
) -> dict:
    """Production ETL pipeline — runs identically in process, Docker, and Kubernetes.

    The work pool type (process/docker/k8s) changes the execution environment;
    the flow code is unchanged.
    """
    raw      = extract(source=source, batch_size=batch_size)
    processed = transform_records(records=raw, scale=scale)
    loaded   = load_to_sink(records=processed, destination=destination)
    summary  = {
        "source":      source,
        "destination": destination,
        "batch_size":  batch_size,
        "loaded":      loaded,
        "scale":       scale,
    }
    print(f"Pipeline summary: {summary}")
    return summary

# Run multiple parameter combinations to simulate different deployment scenarios
with prefect_test_harness():
    print("=" * 60)
    print("Run 1: Default parameters (simulates: process pool, local dev)")
    print("=" * 60)
    r1 = container_etl_pipeline()
    print(f"Result: {r1}")

    print()
    print("=" * 60)
    print("Run 2: Overridden parameters (simulates: Docker pool, staging)")
    print("=" * 60)
    r2 = container_etl_pipeline(
        source="staging-db",
        destination="staging-warehouse.events",
        batch_size=25,
        scale=2.0,
    )
    print(f"Result: {r2}")

    print()
    print("=" * 60)
    print("Run 3: Large batch (simulates: Kubernetes pool, production)")
    print("=" * 60)
    r3 = container_etl_pipeline(
        source="prod-api",
        destination="prod-warehouse.events",
        batch_size=500,
        scale=1.0,
    )
    print(f"Result: {r3}")

print()
print("All three runs used the same flow code — only work pool type and")
print("parameters differ across local/Docker/Kubernetes environments.")


**What just happened?**
- The same flow code runs unchanged across all three pool types (process, Docker, Kubernetes) — this is Prefect's portability guarantee
- Each run had different `batch_size` and `scale` to simulate realistic environment differences (small batch in staging, full batch in prod)
- `prefect_test_harness()` makes this runnable in Colab — in production, remove the context manager and set `PREFECT_API_URL`
- The 3-task structure (extract → transform → load) gives Prefect visibility into each step independently, enabling per-task retry and failure diagnosis


In [ ]:
# Image tag pinning decision helper — enforce the production rule in CI

import re
import sys

def validate_production_image_tag(image: str) -> tuple[bool, str]:
    """Validate that a Docker image tag is safe for production.

    Production images must:
    1. Not use ':latest'
    2. Use a semver or SHA-based tag
    3. Include a registry prefix (not bare Docker Hub)

    Returns (is_valid, reason).
    """
    if ":" not in image:
        return False, "No tag specified — bare image name defaults to :latest"

    repo, tag = image.rsplit(":", 1)

    if tag == "latest":
        return False, ":latest tag is not allowed in production — pin to semver or SHA"

    # Check for semver (X.Y.Z or vX.Y.Z)
    semver_pattern = re.compile(r'^v?\d+\.\d+\.\d+$')
    # Check for SHA-based tag (sha-<hex>)
    sha_pattern = re.compile(r'^sha-[0-9a-f]{7,40}$')
    # Check for date-based tag (YYYY-MM-DD or YYYYMMDD)
    date_pattern = re.compile(r'^\d{4}[-]?\d{2}[-]?\d{2}$')

    if not (semver_pattern.match(tag) or sha_pattern.match(tag) or date_pattern.match(tag)):
        return False, f"Tag '{tag}' is not a recognized stable format (semver / sha-<hex> / YYYYMMDD)"

    # Warn if no registry prefix (bare DockerHub images have supply-chain risk)
    if "/" not in repo:
        return False, f"No registry prefix in '{repo}' — specify a private registry for production"

    return True, f"Tag '{tag}' is production-safe"


# Test with various image references
test_images = [
    "my-flow",                                                                # no tag
    "my-flow:latest",                                                          # :latest — bad
    "my-flow:1.4.2",                                                           # no registry
    "my-flow:dev-branch",                                                      # non-standard tag
    "my-registry.io/flows/my-flow:1.4.2",                                     # good semver
    "us-central1-docker.pkg.dev/proj/flows/my-flow:sha-a3f9b1c0",            # good SHA
    "us-central1-docker.pkg.dev/proj/flows/my-flow:20240115",                 # good date
]

print(f"{'Image':<58} {'Valid':>6}  Reason")
print("-" * 100)
for img in test_images:
    valid, reason = validate_production_image_tag(img)
    status = "✓" if valid else "✗"
    print(f"  {img:<56} {status:>6}  {reason}")


**What just happened?**
- `validate_production_image_tag()` encodes the team's image tagging policy as executable code — run it in CI to gate deployments
- Three accepted formats: semver (`1.4.2`), SHA (`sha-a3f9b1c0`), and date (`20240115`) — all are deterministic and auditable
- Bare DockerHub images (no registry prefix) have supply-chain risk — an attacker who controls the DockerHub account can push a malicious `:latest`
- Add this validator to your `prefect deploy` CI step: if it fails, the pipeline is blocked before the image reaches production


---
## Challenge

You are setting up a production flow that processes sensor readings from IoT devices.

**Requirements:**
- The flow has three tasks: `ingest_sensors`, `filter_outliers`, `write_to_timeseries`
- It should accept `device_group: str`, `window_minutes: int`, `outlier_threshold: float` as parameters
- The `filter_outliers` task should raise a `ValueError` if more than 50% of readings are outliers (value > outlier_threshold)
- Deploy it (build the Deployment manifest, don't apply) targeting a work pool named `"iot-k8s-pool"` with image `"registry.example.com/iot-flow:2.0.1"`

**Your tasks:**
1. Write all three tasks and the flow (use `prefect_test_harness()` for local runs)
2. Run the flow with `device_group="sensor-bank-A"`, `window_minutes=5`, `outlier_threshold=100.0` — should succeed
3. Run the flow with `outlier_threshold=0.5` — should fail (most readings > 0.5)
4. Call `validate_production_image_tag("registry.example.com/iot-flow:2.0.1")` and print the result
5. Generate the K8s Job manifest by calling `generate_k8s_job_manifest()` with the correct arguments and print the `metadata.name` and `spec.template.spec.containers[0].image`


In [ ]:
# Challenge: your solution here
from prefect import flow, task
from prefect.testing.utilities import prefect_test_harness

@task
def ingest_sensors(device_group: str, window_minutes: int) -> list[dict]:
    # YOUR CODE HERE — return simulated sensor readings
    pass

@task
def filter_outliers(readings: list[dict], outlier_threshold: float) -> list[dict]:
    # YOUR CODE HERE — raise ValueError if > 50% are outliers
    pass

@task
def write_to_timeseries(readings: list[dict], device_group: str) -> int:
    # YOUR CODE HERE — return count of records written
    pass

# @flow(name="iot-sensor-pipeline", log_prints=True)
# def iot_pipeline(device_group, window_minutes, outlier_threshold):
#     YOUR CODE HERE

# Step 2: successful run
# YOUR CODE HERE

# Step 3: failing run
# YOUR CODE HERE

# Step 4: validate the image tag
# YOUR CODE HERE

# Step 5: generate K8s Job manifest
# YOUR CODE HERE


---
## Day 12 key concepts recap

| Concept | What to remember |
|---|---|
| Docker work pool | Worker runs containers on a single host; each flow run = one container |
| Kubernetes work pool | Worker submits K8s Jobs; each flow run = one Pod; cluster handles scheduling |
| Image tag pinning | NEVER `:latest` in production — use semver (`1.4.2`) or SHA (`sha-a3f9b1c0`) |
| Multi-stage build | Separate builder and runtime stages — smaller, more secure final image |
| Layer ordering | Dependencies layer before code layer — maximize cache hits on rebuilds |
| `backoffLimit: 0` | K8s should not retry Jobs — let Prefect's `retries` handle it |
| K8s Secrets | Store `PREFECT_API_KEY` as a K8s Secret, not in the Job manifest |
| `auto_remove` / `ttlSecondsAfterFinished` | Clean up stopped containers/Jobs automatically |
| Portability | Same flow code runs in process, Docker, and K8s — only work pool type changes |

> **Tip:** Pin your flow image tag to a specific version (never `:latest` in production) — reproducible container images are the single biggest reliability improvement for production pipelines.

---
## What's next
**Day 13** → Advanced review and capstone: combine everything from Days 1–12 into a multi-environment production pipeline with Docker packaging, Cloud notifications, and full deployment automation.

Mark Day 12 complete in your [tracker](../index.html).
